In [1]:
import json
import numpy as np
import pandas as pd
from functools import reduce
from tqdm import tqdm
import pickle
import io

import mysql.connector
from mysql.connector import MySQLConnection

import larch
from larch.roles import P,X

import pyarrow as pa
import pyarrow.feather as feather

# Data load & preprocess

## Load data

In [2]:
credentials = {
  "user": "external.data.user",
  "password": "wIfQxcjgZ5!gA*FWgi1z$Kj9",
  "host": "35.208.148.42",
  "port": 3306
}

conn: MySQLConnection = mysql.connector.connect(**credentials)

In [3]:
def get_dms_data(conn: MySQLConnection, guid: str):
    cur = conn.cursor()
    query = """
    SELECT DATA FROM external_data.dms WHERE guid = %s
    """

    cur.execute(query, [guid])
    columns = cur.description
    rows = [{columns[index][0]: column for index, column in enumerate(value)} for value in cur.fetchall()]
    cur.close()
    
    return dict(rows[0])['DATA']

def get_dms_data_to_json(conn: MySQLConnection, guid: str):
    return json.loads(get_dms_data(conn, guid))

def get_dms_data_to_df(conn: MySQLConnection, guid: str):
    return feather.read_feather(pa.BufferReader(get_dms_data(conn, guid)))

In [4]:
guid_info = "7c0eea07-99a1-458e-a315-2b67cc9981ad"
guid_topics_dcm = "2ecf3ce4-aede-4516-a049-5d274659e5ec"
guid_genres_dcm = "4a30eb82-dc77-47c0-99ab-98d4ffb9759f"
guid_tags = "73df38cd-31aa-43e5-b05b-6e973ab35295"

In [5]:
info_df = get_dms_data_to_df(conn, guid_info)
topic_dcm_df = get_dms_data_to_df(conn, guid_topics_dcm)
genres_dcm_df = get_dms_data_to_df(conn, guid_genres_dcm)
tags_df = get_dms_data_to_df(conn, guid_tags)

In [6]:
conn.close()

In [7]:
# user metadata1 - age, gender, spend, playtime, loyalty
info_df

,id,survey_meta_id,survey_instance,age,gender,role,favorite_game,spending,playing_time,loyalty
0,1,1,33cbe449-af6a-484b-a521-6a4d08f73454,37,f,10,com.my.defense,75,15,5
1,2,1,5db16995-594b-4b4e-9151-e502b85d6ff3,42,m,10,com.scopely.monopolygo,5,13,4
2,3,1,9722c487-9a13-4a9d-8599-577b69805034,42,f,10,com.dreamgames.royalmatch,5,0,1
3,4,1,7094ec2f-88d4-4642-90f6-f12b41eca4f1,65,m,10,com.samfinaco.paradise,300,15,5
4,5,1,eb6a4d45-c680-4eaf-bef7-aebe5edb7ce0,37,m,10,com.riotgames.league.teamfighttactics,300,8,4
...,...,...,...,...,...,...,...,...,...,...
1965,1966,1,35fc0bbf-cef7-4ad9-b0c7-8bfb50043cc3,32,m,10,com.miniclip.eightballpool,15,8,4
1966,1967,1,b11b6d9c-f739-478a-ab26-d26d11825827,32,m,10,com.nintendo.zara,0,8,1
1967,1968,1,e9c3e858-826f-479e-a85d-6ea3b0e4e43a,32,m,10,com.playtika.wsop.gp,15,8,-1
1968,1969,1,b6e5ec83-0b04-4f2d-8d54-1bc0d6785adc,32,m,4,com.bandainamcogames.dbzdokkanww,0,0,1


In [8]:
# topic - environment + eras / domain + focus / entity / role conjoint
topic_dcm_df

,id,survey_meta_id,survey_instance,dcm_order,topic,feature_1a,feature_1b,feature_2a,feature_2b,chosen_a,chosen_b
0,1,1,33cbe449-af6a-484b-a521-6a4d08f73454,1,topics__fantasy,entities__robot,roles__creator,entities__celestial,roles__team,None,None
1,2,1,33cbe449-af6a-484b-a521-6a4d08f73454,2,topics__fantasy,entities__kid,roles__carer,entities__animal,roles__warrior,None,None
2,3,1,33cbe449-af6a-484b-a521-6a4d08f73454,3,topics__fantasy,environment__aquatic,domains__crime,environment__alternality,domains__business,environment__aquatic,domains__crime
3,4,1,33cbe449-af6a-484b-a521-6a4d08f73454,4,topics__fantasy,entities__adult,roles__rebel,entities__zombie,roles__creator,entities__zombie,roles__creator
4,5,1,33cbe449-af6a-484b-a521-6a4d08f73454,5,topics__fantasy,environment__wilderness,focus__weapons,environment__space,domains__military,None,None
...,...,...,...,...,...,...,...,...,...,...,...
6985,6986,1,8ec75988-2510-4d41-b5da-456e237c2574,1,topics__mystery,entities__celestial,roles__mystic,entities__plant,roles__follower,entities__plant,roles__follower
6986,6987,1,8ec75988-2510-4d41-b5da-456e237c2574,2,topics__mystery,entities__super_,roles__warrior,entities__celestial,roles__creator,entities__super_,roles__warrior
6987,6988,1,8ec75988-2510-4d41-b5da-456e237c2574,3,topics__mystery,entities__human,roles__team,entities__robot,roles__carer,entities__human,roles__team
6988,6989,1,8ec75988-2510-4d41-b5da-456e237c2574,4,topics__mystery,entities__alien,roles__solver,entities__celestial,roles__mystic,entities__alien,roles__solver


In [9]:
# genres - mechanics conjoint
genres_dcm_df

,id,survey_meta_id,survey_instance,dcm_order,genre_1,genre_2,core_1,core_2,chosen
0,1,1,33cbe449-af6a-484b-a521-6a4d08f73454,1,genre__match,None,core__connect,core__merge,core__merge
1,2,1,33cbe449-af6a-484b-a521-6a4d08f73454,2,genre__casino,None,core__slots,core__poker,core__slots
2,3,1,33cbe449-af6a-484b-a521-6a4d08f73454,3,genre__strategy,None,core__4x,core__lane,core__lane
3,4,1,33cbe449-af6a-484b-a521-6a4d08f73454,4,genre__strategy,genre__match,core__lane,core__blast,core__blast
4,5,1,33cbe449-af6a-484b-a521-6a4d08f73454,5,genre__match,genre__strategy,core__merge,core__summon,core__summon
...,...,...,...,...,...,...,...,...,...
8940,8941,1,8ec75988-2510-4d41-b5da-456e237c2574,1,genre__combat,genre__education,core__moba,core__asymetric,core__moba
8941,8942,1,8ec75988-2510-4d41-b5da-456e237c2574,2,genre__education,genre__combat,core__vehicular,core__bullet_hell,core__bullet_hell
8942,8943,1,8ec75988-2510-4d41-b5da-456e237c2574,3,genre__puzzle,genre__casino,core__time_loop,core__poker,core__time_loop
8943,8944,1,8ec75988-2510-4d41-b5da-456e237c2574,4,genre__puzzle,genre__education,core__escape_room,core__move_blocks,core__escape_room


In [10]:
# user metadata 2 - platforms, genres, topics, needs, behaviours, activities, movies, hobbies, socials
tags_df

,id,survey_meta_id,tag_id,tag_value,survey_instance
0,1,1,platforms__mobile,1,33cbe449-af6a-484b-a521-6a4d08f73454
1,2,1,platforms__playstation,1,33cbe449-af6a-484b-a521-6a4d08f73454
2,3,1,platforms__pc,1,33cbe449-af6a-484b-a521-6a4d08f73454
3,4,1,genre__rpg,66,33cbe449-af6a-484b-a521-6a4d08f73454
4,5,1,genre__match,100,33cbe449-af6a-484b-a521-6a4d08f73454
...,...,...,...,...,...
121361,121362,1,multiplayer__multi,4,8ec75988-2510-4d41-b5da-456e237c2574
121362,121363,1,multiplayer__single,3,8ec75988-2510-4d41-b5da-456e237c2574
121363,121364,1,activities__working,1,8ec75988-2510-4d41-b5da-456e237c2574
121364,121365,1,activities__parenting,1,8ec75988-2510-4d41-b5da-456e237c2574


In [11]:
tags_pivot_df = tags_df.pivot(
    index="survey_instance", columns="tag_id", values="tag_value"
).reset_index()

tags_pivot_df.columns.name = None

sorted_categories = ['platforms', 'multiplayer', 'genre', 'topics', 'needs', 'behaviors', 'activities', 'movies', 'hobbies', 'socials']
sorted_col_names = sorted(tags_df.tag_id.unique(), key=lambda x: f"{sorted_categories.index(x.split('_')[0])}_{'_'.join(x.split('_')[2:])}")

tags_pivot_df = tags_pivot_df[['survey_instance'] + sorted_col_names]
tags_pivot_df

,survey_instance,platforms__mobile,platforms__pc,platforms__playstation,platforms__switch,platforms__xbox,multiplayer__multi,multiplayer__single,genre__arcade,genre__building,...,socials__linkedin,socials__pinterest,socials__reddit,socials__snapchat,socials__telegram,socials__tiktok,socials__twitch,socials__twitter,socials__whatsapp,socials__youtube
0,000e7770-bcb0-494f-87b1-ff31f975c98d,1.0,NaN,NaN,NaN,NaN,4.0,4.0,33.0,NaN,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0027a589-c6bf-480d-8e12-77cf79b45152,1.0,NaN,NaN,NaN,NaN,1.0,4.0,100.0,NaN,...,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,NaN
2,003d6fa0-2ce5-4eba-9053-5f0bc832d6ff,1.0,1.0,1.0,1.0,NaN,3.0,4.0,100.0,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,004d89f9-a2ce-4777-b32a-3e111edb5d44,1.0,NaN,NaN,NaN,1.0,3.0,4.0,33.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,005f6277-0f98-417c-86d7-9462a8780587,1.0,NaN,1.0,NaN,1.0,4.0,4.0,100.0,33.0,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1965,ffc684b4-4c08-41c2-9c13-22700afecc6b,1.0,NaN,1.0,NaN,NaN,3.0,4.0,66.0,33.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1966,ffedffd2-21c5-449e-9396-4991f62ca880,1.0,1.0,1.0,NaN,NaN,2.0,4.0,33.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1967,fff0cf1d-11dd-4819-a104-a12051e64a2f,1.0,NaN,1.0,1.0,NaN,3.0,4.0,66.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1968,fff7e72f-0682-4d36-9690-6d8f95466e6a,1.0,1.0,NaN,NaN,1.0,4.0,4.0,66.0,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
sorted_col_names

['platforms__mobile',
 'platforms__pc',
 'platforms__playstation',
 'platforms__switch',
 'platforms__xbox',
 'multiplayer__multi',
 'multiplayer__single',
 'genre__arcade',
 'genre__building',
 'genre__casino',
 'genre__combat',
 'genre__driving',
 'genre__education',
 'genre__idle',
 'genre__match',
 'genre__party',
 'genre__puzzle',
 'genre__rpg',
 'genre__simulation',
 'genre__sports',
 'genre__story',
 'genre__strategy',
 'genre__tabletop',
 'topics__action',
 'topics__adventure',
 'topics__comedy',
 'topics__drama',
 'topics__fantasy',
 'topics__horror',
 'topics__mystery',
 'topics__realistic',
 'topics__romance',
 'topics__sci_fi',
 'needs__acceptance',
 'needs__accomplishment',
 'needs__appreciation',
 'needs__control',
 'needs__fitness',
 'needs__insight',
 'needs__intimacy',
 'needs__mastery',
 'needs__morality',
 'needs__novelty',
 'needs__recognition',
 'needs__relief',
 'needs__safety',
 'needs__superiority',
 'needs__trust',
 'needs__uniqueness',
 'needs__wealth',
 'beha

In [13]:
PLATFORM_COLS = ['platforms__mobile', 'platforms__pc', 'platforms__playstation', 'platforms__switch', 'platforms__xbox']
MULTIPLAYER_COLS = ['multiplayer__multi', 'multiplayer__single']
GENRE_COLS = [
    'genre__arcade', 'genre__building', 'genre__casino', 'genre__combat', 'genre__driving',
    'genre__education', 'genre__idle', 'genre__match', 'genre__party', 'genre__puzzle',
    'genre__rpg', 'genre__simulation', 'genre__sports', 'genre__story', 'genre__strategy',
    'genre__tabletop',
]
TOPIC_COLS = [
    'topics__action', 'topics__adventure', 'topics__comedy', 'topics__drama', 'topics__fantasy','topics__horror', 'topics__mystery', 'topics__realistic', 'topics__romance', 'topics__sci_fi',
]
NEED_COLS = [
    'needs__acceptance', 'needs__accomplishment', 'needs__appreciation', 'needs__control',
    'needs__fitness', 'needs__insight', 'needs__intimacy', 'needs__mastery', 'needs__morality',
    'needs__novelty', 'needs__recognition', 'needs__relief', 'needs__safety', 'needs__superiority',
    'needs__trust', 'needs__uniqueness', 'needs__wealth',
]
BEHAVIOUR_COLS = [
    'behaviors__chill', 'behaviors__collect', 'behaviors__combat', 'behaviors__compete',
    'behaviors__complete', 'behaviors__cooperate', 'behaviors__destroy', 'behaviors__discover',
    'behaviors__empower', 'behaviors__experiment', 'behaviors__express', 'behaviors__immerse',
    'behaviors__plan', 'behaviors__react', 'behaviors__solve',
]
ACTIVITY_COLS = [
    'activities__following_news', 'activities__friends', 'activities__hobbies', 'activities__learning','activities__movies', 'activities__music', 'activities__parenting', 'activities__partner',
    'activities__pets', 'activities__reading', 'activities__social_media', 'activities__video_games',
    'activities__working',
]
MOVIE_COLS = [
    'movies__action_adventure', 'movies__anime', 'movies__children_family', 'movies__classic',
    'movies__comedies', 'movies__criminal', 'movies__documentary', 'movies__drama', 'movies__fantasy',
    'movies__horror', 'movies__reality_shows', 'movies__romantic', 'movies__scifi',
]
HOBBY_COLS = [
    'hobbies__building_fixing', 'hobbies__collecting', 'hobbies__cooking', 'hobbies__design',
    'hobbies__diy', 'hobbies__driving', 'hobbies__exercising_sports', 'hobbies__farm_garden',
    'hobbies__fishing_hunting', 'hobbies__gardening', 'hobbies__performing_arts', 'hobbies__photographing',
    'hobbies__serving_community', 'hobbies__teaching', 'hobbies__watching_games', 'hobbies__watching_sports',
]
SOCIAL_COLS = [
    'socials__discord', 'socials__facebook', 'socials__instagram', 'socials__linkedin', 'socials__pinterest',
    'socials__reddit', 'socials__snapchat', 'socials__telegram', 'socials__tiktok', 'socials__twitch',
    'socials__twitter', 'socials__whatsapp', 'socials__youtube',
]

In [14]:
user_df = info_df.merge(tags_pivot_df, on='survey_instance')

In [15]:
user_df

,id,survey_meta_id,survey_instance,age,gender,role,favorite_game,spending,playing_time,loyalty,...,socials__linkedin,socials__pinterest,socials__reddit,socials__snapchat,socials__telegram,socials__tiktok,socials__twitch,socials__twitter,socials__whatsapp,socials__youtube
0,1,1,33cbe449-af6a-484b-a521-6a4d08f73454,37,f,10,com.my.defense,75,15,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,5db16995-594b-4b4e-9151-e502b85d6ff3,42,m,10,com.scopely.monopolygo,5,13,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,9722c487-9a13-4a9d-8599-577b69805034,42,f,10,com.dreamgames.royalmatch,5,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,7094ec2f-88d4-4642-90f6-f12b41eca4f1,65,m,10,com.samfinaco.paradise,300,15,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,eb6a4d45-c680-4eaf-bef7-aebe5edb7ce0,37,m,10,com.riotgames.league.teamfighttactics,300,8,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1965,1966,1,35fc0bbf-cef7-4ad9-b0c7-8bfb50043cc3,32,m,10,com.miniclip.eightballpool,15,8,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1966,1967,1,b11b6d9c-f739-478a-ab26-d26d11825827,32,m,10,com.nintendo.zara,0,8,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0
1967,1968,1,e9c3e858-826f-479e-a85d-6ea3b0e4e43a,32,m,10,com.playtika.wsop.gp,15,8,-1,...,NaN,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN
1968,1969,1,b6e5ec83-0b04-4f2d-8d54-1bc0d6785adc,32,m,4,com.bandainamcogames.dbzdokkanww,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## User metadata preprocessing

In [16]:
user_df.gender.value_counts()

f    1002
m     945
b      22
o       1
Name: gender, dtype: int64

In [17]:
user_df['gender_female'] = user_df.gender.apply(lambda x: 1 if x == 'f' else 0)

In [18]:
user_df.age.value_counts()

32    345
42    301
21    292
65    277
52    241
37    191
27    177
47    142
18      4
Name: age, dtype: int64

### Platform

In [19]:
for col in PLATFORM_COLS:
    user_df[col] = user_df[col].apply(lambda x: 0 if pd.isna(x) else x)

### Multiplayer

In [20]:
user_df[MULTIPLAYER_COLS].isna().sum()

multiplayer__multi     80
multiplayer__single    80
dtype: int64

In [21]:
user_df[MULTIPLAYER_COLS].mode()

,multiplayer__multi,multiplayer__single
0,3.0,4.0


In [22]:
for col in MULTIPLAYER_COLS:
    user_df[col] = user_df[col].fillna(user_df[col].mode())

### Genres

In [23]:
genres_with_na = user_df[GENRE_COLS].isna().sum().loc[lambda x: x > 0].index.to_list()
GENRE_NA_COLS = [f'{x}_na' for x in genres_with_na]

for col, col_named in zip(genres_with_na, GENRE_NA_COLS):
    if col_named not in user_df:
        user_df[col_named] = user_df[col].apply(lambda x: 1 if pd.isna(x) else 0)
    user_df[col] = user_df[col].apply(lambda x: 0 if pd.isna(x) else x)

### Activity

In [24]:
for col in ACTIVITY_COLS:
    user_df[col] = user_df[col].apply(lambda x: 0 if pd.isna(x) else x)

### Movie

In [25]:
filled_movie = ~((~user_df[MOVIE_COLS].isna()).sum(axis=1) == 0)

for col in MOVIE_COLS:
    user_df.loc[filled_movie, col] = user_df.loc[filled_movie, col].apply(lambda x: 0 if pd.isna(x) else x)

### Hobby

In [26]:
filled_hobby = ~((~user_df[HOBBY_COLS].isna()).sum(axis=1) == 0)

for col in HOBBY_COLS:
    user_df.loc[filled_hobby, col] = user_df.loc[filled_hobby, col].apply(lambda x: 0 if pd.isna(x) else x)

### Social

In [27]:
filled_social = ~((~user_df[SOCIAL_COLS].isna()).sum(axis=1) == 0)

for col in SOCIAL_COLS:
    user_df.loc[filled_social, col] = user_df.loc[filled_social, col].apply(lambda x: 0 if pd.isna(x) else x)

# DCM datasets

## Mechanics

In [28]:
genres_dcm_df

,id,survey_meta_id,survey_instance,dcm_order,genre_1,genre_2,core_1,core_2,chosen
0,1,1,33cbe449-af6a-484b-a521-6a4d08f73454,1,genre__match,None,core__connect,core__merge,core__merge
1,2,1,33cbe449-af6a-484b-a521-6a4d08f73454,2,genre__casino,None,core__slots,core__poker,core__slots
2,3,1,33cbe449-af6a-484b-a521-6a4d08f73454,3,genre__strategy,None,core__4x,core__lane,core__lane
3,4,1,33cbe449-af6a-484b-a521-6a4d08f73454,4,genre__strategy,genre__match,core__lane,core__blast,core__blast
4,5,1,33cbe449-af6a-484b-a521-6a4d08f73454,5,genre__match,genre__strategy,core__merge,core__summon,core__summon
...,...,...,...,...,...,...,...,...,...
8940,8941,1,8ec75988-2510-4d41-b5da-456e237c2574,1,genre__combat,genre__education,core__moba,core__asymetric,core__moba
8941,8942,1,8ec75988-2510-4d41-b5da-456e237c2574,2,genre__education,genre__combat,core__vehicular,core__bullet_hell,core__bullet_hell
8942,8943,1,8ec75988-2510-4d41-b5da-456e237c2574,3,genre__puzzle,genre__casino,core__time_loop,core__poker,core__time_loop
8943,8944,1,8ec75988-2510-4d41-b5da-456e237c2574,4,genre__puzzle,genre__education,core__escape_room,core__move_blocks,core__escape_room


In [29]:
len(genres_dcm_df.chosen.unique())

124

In [30]:
len(pd.concat([genres_dcm_df.core_1, genres_dcm_df.core_2]).unique())

123

In [31]:
genre_mechanic_dict = {}

for col in GENRE_COLS:
    temp_df = genres_dcm_df[(genres_dcm_df.genre_1 == col) & (genres_dcm_df.genre_2.isna())]
    
    genre_mechanics = list(set(pd.concat([temp_df.core_1, temp_df.core_2]).to_list()))
    genre_mechanic_dict[col] = genre_mechanics
    
mechanic_genre_dict = {mechanic: genre for genre, mechanics in genre_mechanic_dict.items() for mechanic in mechanics}

mechanic_idx_dict = {x: idx for idx, x in enumerate(mechanic_genre_dict)}
idx_mechanic_dict = {y: x for x, y in mechanic_idx_dict.items()}
    
len(mechanic_genre_dict)

123

In [32]:
cols_used = ['age'] + GENRE_COLS + GENRE_NA_COLS + TOPIC_COLS + BEHAVIOUR_COLS # 'gender_female', 

df_mech = genres_dcm_df.merge(user_df, on='survey_instance')
av_cols = []

# availablity cols
for idx, col in idx_mechanic_dict.items():
    av_col = ((df_mech.core_1 == col).astype(int) + (df_mech.core_2 == col).astype(int))
    av_col.name = f'av_{idx}'
    
    av_cols.append(av_col)

df_mech = pd.concat([df_mech] + av_cols, axis=1).copy()
df_mech[f'av_{len(idx_mechanic_dict)}'] = 1
df_mech['choice'] = df_mech['chosen'].apply(lambda x: len(idx_mechanic_dict) if x is None else mechanic_idx_dict[x])

# combined cols
new_dcm_cols = []
idx_cols_dict = {x:[] for x in idx_mechanic_dict}

for user_col in tqdm(cols_used):
    user_col_min = df_mech[user_col].min()
    user_col_max = df_mech[user_col].max()
    
    for idx, col in idx_mechanic_dict.items():
        dcm_user_col = df_mech[f'av_{idx}'] * df_mech[user_col]
        # min-max scaling -> 0-1
        dcm_user_col = (dcm_user_col - user_col_min) / (user_col_max - user_col_min)
        dcm_user_col.name = f'{col}__{user_col}'

        idx_cols_dict[idx].append(f'{col}__{user_col}')
        new_dcm_cols.append(dcm_user_col)

df_mech = pd.concat([df_mech] + new_dcm_cols, axis=1).copy()

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:02<00:00, 21.53it/s]


In [33]:
m = larch.Model()

In [34]:
m.choice_co_code = 'choice'

In [35]:
m.availability_co_vars = {idx: f'av_{idx}' for idx in range(len(idx_mechanic_dict) + 1)}

for idx, user_cols in idx_cols_dict.items():
    m.utility_co[idx] = P(f'{idx_mechanic_dict[idx]}_ASC')
    
    for user_col in user_cols:
        m.utility_co[idx] += X(user_col) * P(user_col)

m.utility_co[len(idx_mechanic_dict)] = 0

dfs = larch.DataFrames(df_mech, alt_codes=[i for i in range(len(idx_mechanic_dict) + 1)])

In [36]:
m.dataservice = dfs

In [37]:
m.load_data()

C:\Users\daide\AppData\Local\Temp/ipykernel_21908/1940460508.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  m.load_data()


In [38]:
%%time
m.maximize_loglike(quiet=False, maxiter=30)

,value,initvalue,nullvalue,minimum,maximum,holdfast,note,best
core__4x_ASC,-0.288217,0.0,0.0,-inf,inf,0,,-0.288136
core__4x__age,-0.269079,0.0,0.0,-inf,inf,0,,-0.269130
core__4x__behaviors__chill,-0.721520,0.0,0.0,-inf,inf,0,,-0.721916
core__4x__behaviors__collect,-0.445766,0.0,0.0,-inf,inf,0,,-0.445799
core__4x__behaviors__combat,0.591848,0.0,0.0,-inf,inf,0,,0.590490
...,...,...,...,...,...,...,...,...
core__word__topics__horror,0.368188,0.0,0.0,-inf,inf,0,,0.368101
core__word__topics__mystery,-0.540075,0.0,0.0,-inf,inf,0,,-0.540396
core__word__topics__realistic,0.800750,0.0,0.0,-inf,inf,0,,0.800921
core__word__topics__romance,-0.255829,0.0,0.0,-inf,inf,0,,-0.256592


c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 5.242305509795312e-32 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 5.318516118277616e-32 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 2.022211352206651e-32 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 2.0743517216189916e-32 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 1.4328374080837507e-32 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\minico

KeyboardInterrupt: 

In [39]:
m.get_values()

core__4x_ASC                    -0.288217
core__4x__age                   -0.269079
core__4x__behaviors__chill      -0.721520
core__4x__behaviors__collect    -0.445766
core__4x__behaviors__combat      0.591848
                                   ...   
core__word__topics__horror       0.368188
core__word__topics__mystery     -0.540075
core__word__topics__realistic    0.800750
core__word__topics__romance     -0.255829
core__word__topics__sci_fi      -0.400303
Name: value, Length: 6273, dtype: float64

### Results

In [40]:
pickle.dump(m.get_values().to_dict(), open("mechanic_coefs_2.p", "wb"))

In [41]:
pickle.load(open("mechanic_coefs_2.p", "rb"))

{'core__4x_ASC': -0.28821734003813154,
 'core__4x__age': -0.26907939704942685,
 'core__4x__behaviors__chill': -0.721519508567868,
 'core__4x__behaviors__collect': -0.44576552105507145,
 'core__4x__behaviors__combat': 0.5918475969497278,
 'core__4x__behaviors__compete': -0.2732887488012513,
 'core__4x__behaviors__complete': -0.04599465094773169,
 'core__4x__behaviors__cooperate': -0.08774136279446082,
 'core__4x__behaviors__destroy': 0.05689770468239871,
 'core__4x__behaviors__discover': 0.31786734080878,
 'core__4x__behaviors__empower': 0.8337100954705845,
 'core__4x__behaviors__experiment': -0.21595277706346722,
 'core__4x__behaviors__express': -0.07542662979010684,
 'core__4x__behaviors__immerse': 0.7484224865352311,
 'core__4x__behaviors__plan': 0.5829705553833191,
 'core__4x__behaviors__react': -0.36743293941658983,
 'core__4x__behaviors__solve': 0.2875541709602418,
 'core__4x__genre__arcade': 0.7103442010168542,
 'core__4x__genre__building': 3.111070677803588,
 'core__4x__genre__b

## Topics - 4 datasets

In [42]:
topic_dcm_df

,id,survey_meta_id,survey_instance,dcm_order,topic,feature_1a,feature_1b,feature_2a,feature_2b,chosen_a,chosen_b
0,1,1,33cbe449-af6a-484b-a521-6a4d08f73454,1,topics__fantasy,entities__robot,roles__creator,entities__celestial,roles__team,None,None
1,2,1,33cbe449-af6a-484b-a521-6a4d08f73454,2,topics__fantasy,entities__kid,roles__carer,entities__animal,roles__warrior,None,None
2,3,1,33cbe449-af6a-484b-a521-6a4d08f73454,3,topics__fantasy,environment__aquatic,domains__crime,environment__alternality,domains__business,environment__aquatic,domains__crime
3,4,1,33cbe449-af6a-484b-a521-6a4d08f73454,4,topics__fantasy,entities__adult,roles__rebel,entities__zombie,roles__creator,entities__zombie,roles__creator
4,5,1,33cbe449-af6a-484b-a521-6a4d08f73454,5,topics__fantasy,environment__wilderness,focus__weapons,environment__space,domains__military,None,None
...,...,...,...,...,...,...,...,...,...,...,...
6985,6986,1,8ec75988-2510-4d41-b5da-456e237c2574,1,topics__mystery,entities__celestial,roles__mystic,entities__plant,roles__follower,entities__plant,roles__follower
6986,6987,1,8ec75988-2510-4d41-b5da-456e237c2574,2,topics__mystery,entities__super_,roles__warrior,entities__celestial,roles__creator,entities__super_,roles__warrior
6987,6988,1,8ec75988-2510-4d41-b5da-456e237c2574,3,topics__mystery,entities__human,roles__team,entities__robot,roles__carer,entities__human,roles__team
6988,6989,1,8ec75988-2510-4d41-b5da-456e237c2574,4,topics__mystery,entities__alien,roles__solver,entities__celestial,roles__mystic,entities__alien,roles__solver


In [64]:
# Entity / era + enviro
topic_dcm_df.feature_1a.unique()

array(['entities__robot', 'entities__kid', 'environment__aquatic',
       'entities__adult', 'environment__wilderness', 'entities__alien',
       'eras__prehistoric', 'eras__medieval', 'entities__zombie',
       'entities__elder', 'eras__near_future', 'environment__alternality',
       'entities__super_', 'environment__space', 'eras__industrial',
       'environment__suburban', 'entities__human', 'entities__celestial',
       'eras__renaissance', 'eras__ancient', 'entities__microbe',
       'eras__timeless', 'eras__modern', 'entities__plant',
       'entities__monster', 'environment__rural', 'entities__animal',
       'environment__virtuality', 'eras__retro', 'environment__urban'],
      dtype=object)

In [65]:
# Role / domain + focus
topic_dcm_df.feature_1b.unique()

array(['roles__creator', 'roles__carer', 'domains__crime', 'roles__rebel',
       'focus__weapons', 'roles__authority', 'focus__structures',
       'domains__military', 'roles__solver', 'domains__food',
       'roles__explorer', 'domains__health', 'domains__art',
       'roles__team', 'roles__follower', 'roles__mystic',
       'focus__solutions', 'domains__luxury', 'focus__levels',
       'focus__events', 'roles__warrior', 'focus__cities',
       'domains__sports', 'domains__fashion', 'domains__rights',
       'focus__actors', 'domains__religion', 'domains__nature',
       'focus__vehicles', 'focus__interiors', 'roles__villain',
       'domains__politics', 'focus__bases', 'focus__self',
       'domains__family', 'focus__objects', 'domains__mythology',
       'domains__business'], dtype=object)

### Split datasets

In [66]:
dataset_split = topic_dcm_df.feature_1a.apply(lambda x: x.startswith('entities__'))
dataset_split

0        True
1        True
2       False
3        True
4       False
        ...  
6985     True
6986     True
6987     True
6988     True
6989    False
Name: feature_1a, Length: 6990, dtype: bool

In [67]:
topic_dcm_df1 = topic_dcm_df[dataset_split].copy()
topic_dcm_df2 = topic_dcm_df[~dataset_split].copy()

In [68]:
entity_idx_dict = {x: idx for idx, x in enumerate(sorted(topic_dcm_df1.feature_1a.unique()))}
idx_entity_dict = {y: x for x, y in entity_idx_dict.items()}

role_idx_dict = {x: idx for idx, x in enumerate(sorted(topic_dcm_df1.feature_1b.unique()))}
idx_role_dict = {y: x for x, y in role_idx_dict.items()}

era_enviro_idx_dict = {x: idx for idx, x in enumerate(sorted(topic_dcm_df2.feature_1a.unique()))}
idx_era_enviro_dict = {y: x for x, y in era_enviro_idx_dict.items()}

domain_focus_idx_dict = {x: idx for idx, x in enumerate(sorted(topic_dcm_df2.feature_1b.unique()))}
idx_domain_focus_dict = {y: x for x, y in domain_focus_idx_dict.items()}

## Entity

In [69]:
cols_used = ['age'] + GENRE_COLS + GENRE_NA_COLS + TOPIC_COLS + BEHAVIOUR_COLS # 'gender_female', 

df_ent = topic_dcm_df1.merge(user_df, on='survey_instance')
av_cols = []

# availablity cols
for idx, col in idx_entity_dict.items():
    av_col = ((df_ent.feature_1a == col).astype(int) + (df_ent.feature_2a == col).astype(int))
    av_col.name = f'av_{idx}'
    
    av_cols.append(av_col)

df_ent = pd.concat([df_ent] + av_cols, axis=1).copy()
df_ent[f'av_{len(idx_entity_dict)}'] = 1
df_ent['choice'] = df_ent['chosen_a'].apply(lambda x: len(idx_entity_dict) if x is None else entity_idx_dict[x])

# combined cols
new_dcm_cols = []
idx_cols_dict = {x:[] for x in idx_entity_dict}

for user_col in tqdm(cols_used):
    user_col_min = df_ent[user_col].min()
    user_col_max = df_ent[user_col].max()
    
    for idx, col in idx_entity_dict.items():
        dcm_user_col = df_ent[f'av_{idx}'] * df_ent[user_col]
        # min-max scaling -> 0-1
        dcm_user_col = (dcm_user_col - user_col_min) / (user_col_max - user_col_min)
        dcm_user_col.name = f'{col}__{user_col}'

        idx_cols_dict[idx].append(f'{col}__{user_col}')
        new_dcm_cols.append(dcm_user_col)

df_ent = pd.concat([df_ent] + new_dcm_cols, axis=1).copy()

100%|█████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 248.50it/s]


In [70]:
m = larch.Model()

In [71]:
m.choice_co_code = 'choice'

In [72]:
m.availability_co_vars = {idx: f'av_{idx}' for idx in range(len(idx_entity_dict) + 1)}

for idx, user_cols in idx_cols_dict.items():
    m.utility_co[idx] = P(f'{idx_entity_dict[idx]}_ASC')
    
    for user_col in user_cols:
        m.utility_co[idx] += X(user_col) * P(user_col)

m.utility_co[len(idx_entity_dict)] = 0

dfs = larch.DataFrames(df_ent, alt_codes=[i for i in range(len(idx_entity_dict) + 1)])

In [73]:
m.dataservice = dfs

In [74]:
m.load_data()

C:\Users\daide\AppData\Local\Temp/ipykernel_12476/1940460508.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  m.load_data()


In [75]:
%%time
m.maximize_loglike(quiet=False)

,value,initvalue,nullvalue,minimum,maximum,holdfast,note,best
entities__adult_ASC,-89.180184,0.0,0.0,-inf,inf,0,,-89.180184
entities__adult__age,0.749316,0.0,0.0,-inf,inf,0,,0.749316
entities__adult__behaviors__chill,1.616049,0.0,0.0,-inf,inf,0,,1.616049
entities__adult__behaviors__collect,-0.520942,0.0,0.0,-inf,inf,0,,-0.520942
entities__adult__behaviors__combat,-0.214125,0.0,0.0,-inf,inf,0,,-0.214125
...,...,...,...,...,...,...,...,...
entities__zombie__topics__horror,1.655545,0.0,0.0,-inf,inf,0,,1.655545
entities__zombie__topics__mystery,0.097042,0.0,0.0,-inf,inf,0,,0.097042
entities__zombie__topics__realistic,0.291333,0.0,0.0,-inf,inf,0,,0.291333
entities__zombie__topics__romance,-0.753396,0.0,0.0,-inf,inf,0,,-0.753396


c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 9.988205314391124e-06 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 2.7755591486768624e-14 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 3.4025100042990448e-15 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 3.7377880584179465e-15 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 2.388251871565744e-15 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\minic

Wall time: 2min


┣          loglike: -2112.6262801781268
┣                x: entities__adult_ASC                   -89.180184
┃                   entities__adult__age                    0.749316
┃                   entities__adult__behaviors__chill       1.616049
┃                   entities__adult__behaviors__collect    -0.520942
┃                   entities__adult__behaviors__combat     -0.214125
┃                                                            ...    
┃                   entities__zombie__topics__horror        1.655545
┃                   entities__zombie__topics__mystery       0.097042
┃                   entities__zombie__topics__realistic     0.291333
┃                   entities__zombie__topics__romance      -0.753396
┃                   entities__zombie__topics__sci_fi        2.207079
┃                   Length: 663, dtype: float64
┣        tolerance: 9.584537943065642e-06
┣            steps: array([1.   , 1.   , 0.5  , 0.5  , 0.5  , 0.5  , 0.5  , 0.5  , 0.5  ,
┃                          0.5  , 0.25 , 0.25 , 0.25 , 0.25 , 0.25 , 0.25 , 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125])
┣          message: 'Optimization terminated successfully.'
┣     elapsed_time: datetime.timedelta(seconds=120, microseconds=368409)
┣           method: 'bhhh'
┣          n_cases: 3046
┣ iteration_number: 95
┣          logloss: 0.6935739593493522

### Results

In [76]:
pickle.dump(m.get_values().to_dict(), open("entity_coefs_2.p", "wb"))

In [77]:
pickle.load(open("entity_coefs_2.p", "rb"))

{'entities__adult_ASC': -89.18018371080203,
 'entities__adult__age': 0.7493159712045754,
 'entities__adult__behaviors__chill': 1.6160489878049307,
 'entities__adult__behaviors__collect': -0.5209415571477631,
 'entities__adult__behaviors__combat': -0.2141252634882337,
 'entities__adult__behaviors__compete': 0.6482232938775078,
 'entities__adult__behaviors__complete': -1.9773429660757311,
 'entities__adult__behaviors__cooperate': 1.5110483266727397,
 'entities__adult__behaviors__destroy': -0.03727283801361043,
 'entities__adult__behaviors__discover': -0.19925106992945502,
 'entities__adult__behaviors__empower': 1.5560942855725797,
 'entities__adult__behaviors__experiment': 0.10514911892305662,
 'entities__adult__behaviors__express': -1.660219047857826,
 'entities__adult__behaviors__immerse': 0.1972442165887669,
 'entities__adult__behaviors__plan': 0.14392805862872293,
 'entities__adult__behaviors__react': 0.3033338668309503,
 'entities__adult__behaviors__solve': 0.7976565439879272,
 'ent

## Role

In [78]:
cols_used = ['age'] + GENRE_COLS + GENRE_NA_COLS + TOPIC_COLS + BEHAVIOUR_COLS # 'gender_female', 

df_role = topic_dcm_df1.merge(user_df, on='survey_instance')
av_cols = []

# availablity cols
for idx, col in idx_role_dict.items():
    av_col = ((df_role.feature_1b == col).astype(int) + (df_role.feature_2b == col).astype(int))
    av_col.name = f'av_{idx}'
    
    av_cols.append(av_col)

df_role = pd.concat([df_role] + av_cols, axis=1).copy()
df_role[f'av_{len(idx_role_dict)}'] = 1
df_role['choice'] = df_role['chosen_b'].apply(lambda x: len(idx_role_dict) if x is None else role_idx_dict[x])

# combined cols
new_dcm_cols = []
idx_cols_dict = {x:[] for x in idx_role_dict}

for user_col in tqdm(cols_used):
    user_col_min = df_role[user_col].min()
    user_col_max = df_role[user_col].max()
    
    for idx, col in idx_role_dict.items():
        dcm_user_col = df_role[f'av_{idx}'] * df_role[user_col]
        # min-max scaling -> 0-1
        dcm_user_col = (dcm_user_col - user_col_min) / (user_col_max - user_col_min)
        dcm_user_col.name = f'{col}__{user_col}'

        idx_cols_dict[idx].append(f'{col}__{user_col}')
        new_dcm_cols.append(dcm_user_col)

df_role = pd.concat([df_role] + new_dcm_cols, axis=1).copy()

100%|█████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 268.16it/s]


In [79]:
m = larch.Model()

In [80]:
m.choice_co_code = 'choice'

In [81]:
m.availability_co_vars = {idx: f'av_{idx}' for idx in range(len(idx_role_dict) + 1)}

for idx, user_cols in idx_cols_dict.items():
    m.utility_co[idx] = P(f'{idx_role_dict[idx]}_ASC')
    
    for user_col in user_cols:
        m.utility_co[idx] += X(user_col) * P(user_col)

m.utility_co[len(idx_role_dict)] = 0

dfs = larch.DataFrames(df_role, alt_codes=[i for i in range(len(idx_role_dict) + 1)])

In [82]:
m.dataservice = dfs

In [83]:
m.load_data()

C:\Users\daide\AppData\Local\Temp/ipykernel_12476/1940460508.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  m.load_data()


In [84]:
%%time
m.maximize_loglike(quiet=False)

,value,initvalue,nullvalue,minimum,maximum,holdfast,note,best
roles__authority_ASC,0.913807,0.0,0.0,-inf,inf,0,,0.913807
roles__authority__age,0.942196,0.0,0.0,-inf,inf,0,,0.942196
roles__authority__behaviors__chill,0.392676,0.0,0.0,-inf,inf,0,,0.392676
roles__authority__behaviors__collect,-0.481727,0.0,0.0,-inf,inf,0,,-0.481727
roles__authority__behaviors__combat,0.068560,0.0,0.0,-inf,inf,0,,0.068560
...,...,...,...,...,...,...,...,...
roles__warrior__topics__horror,1.521173,0.0,0.0,-inf,inf,0,,1.521173
roles__warrior__topics__mystery,-0.029274,0.0,0.0,-inf,inf,0,,-0.029274
roles__warrior__topics__realistic,0.000764,0.0,0.0,-inf,inf,0,,0.000764
roles__warrior__topics__romance,-0.020722,0.0,0.0,-inf,inf,0,,-0.020722


Wall time: 31.1 s


┣          loglike: -2304.4261458781425
┣                x: roles__authority_ASC                    0.913807
┃                   roles__authority__age                   0.942196
┃                   roles__authority__behaviors__chill      0.392676
┃                   roles__authority__behaviors__collect   -0.481727
┃                   roles__authority__behaviors__combat     0.068560
┃                                                             ...   
┃                   roles__warrior__topics__horror          1.521173
┃                   roles__warrior__topics__mystery        -0.029274
┃                   roles__warrior__topics__realistic       0.000764
┃                   roles__warrior__topics__romance        -0.020722
┃                   roles__warrior__topics__sci_fi          1.282671
┃                   Length: 561, dtype: float64
┣        tolerance: 8.29056506938578e-06
┣            steps: array([1. , 1. , 1. , 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5,
┃                          0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5,
┃                          0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])
┣          message: 'Optimization terminated successfully.'
┣     elapsed_time: datetime.timedelta(seconds=31, microseconds=120283)
┣           method: 'bhhh'
┣          n_cases: 3046
┣ iteration_number: 39
┣          logloss: 0.7565417419166587

### Results

In [85]:
pickle.dump(m.get_values().to_dict(), open("role_coefs_2.p", "wb"))

In [86]:
pickle.load(open("role_coefs_2.p", "rb"))

{'roles__authority_ASC': 0.9138067835247548,
 'roles__authority__age': 0.9421963787296576,
 'roles__authority__behaviors__chill': 0.3926756142927551,
 'roles__authority__behaviors__collect': -0.4817273154693917,
 'roles__authority__behaviors__combat': 0.06855975799651985,
 'roles__authority__behaviors__compete': 0.0949355360186911,
 'roles__authority__behaviors__complete': -0.5780808601388445,
 'roles__authority__behaviors__cooperate': 0.922573883135487,
 'roles__authority__behaviors__destroy': 0.5514492090291896,
 'roles__authority__behaviors__discover': -0.07255393697383873,
 'roles__authority__behaviors__empower': 0.4359040139474411,
 'roles__authority__behaviors__experiment': 0.6893595932260929,
 'roles__authority__behaviors__express': -0.4024283665302244,
 'roles__authority__behaviors__immerse': 0.025990116100552126,
 'roles__authority__behaviors__plan': 0.49190015644428114,
 'roles__authority__behaviors__react': 0.0443734322535008,
 'roles__authority__behaviors__solve': -0.225847

## Environment + eras

In [87]:
cols_used = ['age'] + GENRE_COLS + GENRE_NA_COLS + TOPIC_COLS + BEHAVIOUR_COLS # 'gender_female', 

df_era = topic_dcm_df2.merge(user_df, on='survey_instance')
av_cols = []

# availablity cols
for idx, col in idx_era_enviro_dict.items():
    av_col = ((df_era.feature_1a == col).astype(int) + (df_era.feature_2a == col).astype(int))
    av_col.name = f'av_{idx}'
    
    av_cols.append(av_col)

df_era = pd.concat([df_era] + av_cols, axis=1).copy()
df_era[f'av_{len(idx_era_enviro_dict)}'] = 1
df_era['choice'] = df_era['chosen_a'].apply(lambda x: len(idx_era_enviro_dict) if x is None else era_enviro_idx_dict[x])

# combined cols
new_dcm_cols = []
idx_cols_dict = {x:[] for x in idx_era_enviro_dict}

for user_col in tqdm(cols_used):
    user_col_min = df_era[user_col].min()
    user_col_max = df_era[user_col].max()
    
    for idx, col in idx_era_enviro_dict.items():
        dcm_user_col = df_era[f'av_{idx}'] * df_era[user_col]
        # min-max scaling -> 0-1
        dcm_user_col = (dcm_user_col - user_col_min) / (user_col_max - user_col_min)
        dcm_user_col.name = f'{col}__{user_col}'

        idx_cols_dict[idx].append(f'{col}__{user_col}')
        new_dcm_cols.append(dcm_user_col)

df_era = pd.concat([df_era] + new_dcm_cols, axis=1).copy()

100%|█████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 194.71it/s]


In [88]:
m = larch.Model()

In [89]:
m.choice_co_code = 'choice'

In [90]:
m.availability_co_vars = {idx: f'av_{idx}' for idx in range(len(idx_era_enviro_dict) + 1)}

for idx, user_cols in idx_cols_dict.items():
    m.utility_co[idx] = P(f'{idx_era_enviro_dict[idx]}_ASC')
    
    for user_col in user_cols:
        m.utility_co[idx] += X(user_col) * P(user_col)

m.utility_co[len(idx_era_enviro_dict)] = 0

dfs = larch.DataFrames(df_era, alt_codes=[i for i in range(len(idx_era_enviro_dict) + 1)])

In [91]:
m.dataservice = dfs

In [92]:
m.load_data()

C:\Users\daide\AppData\Local\Temp/ipykernel_12476/1940460508.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  m.load_data()


In [93]:
%%time
m.maximize_loglike(quiet=False)

,value,initvalue,nullvalue,minimum,maximum,holdfast,note,best
environment__alternality_ASC,-4.566236,0.0,0.0,-inf,inf,0,,-4.566236
environment__alternality__age,-0.960144,0.0,0.0,-inf,inf,0,,-0.960144
environment__alternality__behaviors__chill,0.961490,0.0,0.0,-inf,inf,0,,0.961490
environment__alternality__behaviors__collect,-0.464641,0.0,0.0,-inf,inf,0,,-0.464641
environment__alternality__behaviors__combat,-0.403260,0.0,0.0,-inf,inf,0,,-0.403260
...,...,...,...,...,...,...,...,...
eras__timeless__topics__horror,0.397137,0.0,0.0,-inf,inf,0,,0.397137
eras__timeless__topics__mystery,-0.041323,0.0,0.0,-inf,inf,0,,-0.041323
eras__timeless__topics__realistic,0.972961,0.0,0.0,-inf,inf,0,,0.972961
eras__timeless__topics__romance,0.035081,0.0,0.0,-inf,inf,0,,0.035081


c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 3.518383484961034e-11 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 1.6775913991295965e-11 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 1.4695102913546064e-11 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 1.3168312646458168e-11 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 1.2050499435420207e-11 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\mini

Wall time: 4min 24s


┣          loglike: -2902.399422545532
┣                x: environment__alternality_ASC                   -4.566236
┃                   environment__alternality__age                  -0.960144
┃                   environment__alternality__behaviors__chill      0.961490
┃                   environment__alternality__behaviors__collect   -0.464641
┃                   environment__alternality__behaviors__combat    -0.403260
┃                                                                     ...   
┃                   eras__timeless__topics__horror                  0.397137
┃                   eras__timeless__topics__mystery                -0.041323
┃                   eras__timeless__topics__realistic               0.972961
┃                   eras__timeless__topics__romance                 0.035081
┃                   eras__timeless__topics__sci_fi                  1.536521
┃                   Length: 867, dtype: float64
┣        tolerance: 0.021836405302391792
┣            steps: array([1.     , 1.     , 0.5    , 0.5    , 0.25   , 0.25   , 0.25   ,
┃                          0.25   , 0.25   , 0.25   , 0.25   , 0.25   , 0.125  , 0.0625 ,
┃                          0.0625 , 0.0625 , 0.0625 , 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
┃                          0.03125, 0.03125])
┣          message: 'Optimization terminated after 100 iterations.'
┣     elapsed_time: datetime.timedelta(seconds=264, microseconds=275742)
┣           method: 'bhhh'
┣          n_cases: 3944
┣ iteration_number: 100
┣          logloss: 0.735902490503431

### Results

In [94]:
pickle.dump(m.get_values().to_dict(), open("enviro_era_coefs_2.p", "wb"))

In [95]:
pickle.load(open("enviro_era_coefs_2.p", "rb"))

{'environment__alternality_ASC': -4.566235690866905,
 'environment__alternality__age': -0.9601435773171749,
 'environment__alternality__behaviors__chill': 0.9614899119755768,
 'environment__alternality__behaviors__collect': -0.4646409935638142,
 'environment__alternality__behaviors__combat': -0.4032600931290022,
 'environment__alternality__behaviors__compete': 0.2931782791311439,
 'environment__alternality__behaviors__complete': 0.7521686028547492,
 'environment__alternality__behaviors__cooperate': 0.00535168893649527,
 'environment__alternality__behaviors__destroy': 1.044303992859118,
 'environment__alternality__behaviors__discover': 1.0792565463169208,
 'environment__alternality__behaviors__empower': -0.4417815853148955,
 'environment__alternality__behaviors__experiment': -0.10866189320494932,
 'environment__alternality__behaviors__express': 0.05309379375952228,
 'environment__alternality__behaviors__immerse': 0.29571392959221,
 'environment__alternality__behaviors__plan': 0.30573580

## Focus + domain

In [96]:
cols_used = ['age'] + GENRE_COLS + GENRE_NA_COLS + TOPIC_COLS + BEHAVIOUR_COLS # 'gender_female', 

df_focus = topic_dcm_df2.merge(user_df, on='survey_instance')
av_cols = []

# availablity cols
for idx, col in idx_domain_focus_dict.items():
    av_col = ((df_focus.feature_1b == col).astype(int) + (df_focus.feature_2b == col).astype(int))
    av_col.name = f'av_{idx}'
    
    av_cols.append(av_col)

df_focus = pd.concat([df_focus] + av_cols, axis=1).copy()
df_focus[f'av_{len(idx_domain_focus_dict)}'] = 1
df_focus['choice'] = df_focus['chosen_b'].apply(lambda x: len(idx_domain_focus_dict) if x is None else domain_focus_idx_dict[x])

# combined cols
new_dcm_cols = []
idx_cols_dict = {x:[] for x in idx_domain_focus_dict}

for user_col in tqdm(cols_used):
    user_col_min = df_focus[user_col].min()
    user_col_max = df_focus[user_col].max()
    
    for idx, col in idx_domain_focus_dict.items():
        dcm_user_col = df_focus[f'av_{idx}'] * df_focus[user_col]
        # min-max scaling -> 0-1
        dcm_user_col = (dcm_user_col - user_col_min) / (user_col_max - user_col_min)
        dcm_user_col.name = f'{col}__{user_col}'

        idx_cols_dict[idx].append(f'{col}__{user_col}')
        new_dcm_cols.append(dcm_user_col)

df_focus = pd.concat([df_focus] + new_dcm_cols, axis=1).copy()

100%|█████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 115.85it/s]


In [97]:
m = larch.Model()

In [98]:
m.choice_co_code = 'choice'

In [99]:
m.availability_co_vars = {idx: f'av_{idx}' for idx in range(len(idx_domain_focus_dict) + 1)}

for idx, user_cols in idx_cols_dict.items():
    m.utility_co[idx] = P(f'{idx_domain_focus_dict[idx]}_ASC')
    
    for user_col in user_cols:
        m.utility_co[idx] += X(user_col) * P(user_col)

m.utility_co[len(idx_domain_focus_dict)] = 0

dfs = larch.DataFrames(df_focus, alt_codes=[i for i in range(len(idx_domain_focus_dict) + 1)])

In [100]:
m.dataservice = dfs

In [101]:
m.load_data()

C:\Users\daide\AppData\Local\Temp/ipykernel_12476/1940460508.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  m.load_data()


In [102]:
%%time
m.maximize_loglike(quiet=False)

,value,initvalue,nullvalue,minimum,maximum,holdfast,note,best
domains__art_ASC,-3435.003401,0.0,0.0,-inf,inf,0,,-3435.003401
domains__art__age,-0.122020,0.0,0.0,-inf,inf,0,,-0.122020
domains__art__behaviors__chill,1.253261,0.0,0.0,-inf,inf,0,,1.253261
domains__art__behaviors__collect,0.083912,0.0,0.0,-inf,inf,0,,0.083912
domains__art__behaviors__combat,0.093450,0.0,0.0,-inf,inf,0,,0.093450
...,...,...,...,...,...,...,...,...
focus__weapons__topics__horror,0.911709,0.0,0.0,-inf,inf,0,,0.911709
focus__weapons__topics__mystery,0.296231,0.0,0.0,-inf,inf,0,,0.296231
focus__weapons__topics__realistic,-0.598322,0.0,0.0,-inf,inf,0,,-0.598322
focus__weapons__topics__romance,-1.859022,0.0,0.0,-inf,inf,0,,-1.859022


c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 7.744639727613547e-10 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 5.485612065125567e-17 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 1.195003410181122e-15 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 5.414207906356572e-17 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\miniconda\lib\site-packages\larch\linalg\__init__.py:18: UserWarning: minimum eig 1.5058899332260185e-16 in general_inverse
  warnings.warn(f"minimum eig {min_eig} in general_inverse")
c:\programs\minicon

Wall time: 11min 13s


┣          loglike: -2456.9435908446926
┣                x: domains__art_ASC                    -3435.003401
┃                   domains__art__age                      -0.122020
┃                   domains__art__behaviors__chill          1.253261
┃                   domains__art__behaviors__collect        0.083912
┃                   domains__art__behaviors__combat         0.093450
┃                                                           ...     
┃                   focus__weapons__topics__horror          0.911709
┃                   focus__weapons__topics__mystery         0.296231
┃                   focus__weapons__topics__realistic      -0.598322
┃                   focus__weapons__topics__romance        -1.859022
┃                   focus__weapons__topics__sci_fi          2.726362
┃                   Length: 1377, dtype: float64
┣        tolerance: 0.0004542438794328346
┣            steps: array([1.   , 1.   , 0.5  , 0.5  , 0.25 , 0.25 , 0.25 , 0.25 , 0.25 ,
┃                          0.25 , 0.25 , 0.25 , 0.25 , 0.25 , 0.25 , 0.25 , 0.25 , 0.25 ,
┃                          0.25 , 0.25 , 0.25 , 0.25 , 0.25 , 0.25 , 0.25 , 0.25 , 0.25 ,
┃                          0.25 , 0.25 , 0.25 , 0.25 , 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125,
┃                          0.125])
┣          message: 'Optimization terminated after 100 iterations.'
┣     elapsed_time: datetime.timedelta(seconds=673, microseconds=406456)
┣           method: 'bhhh'
┣          n_cases: 3944
┣ iteration_number: 100
┣          logloss: 0.6229572999099119

### Results

In [103]:
pickle.dump(m.get_values().to_dict(), open("focus_domain_coefs_2.p", "wb"))

In [104]:
pickle.load(open("focus_domain_coefs_2.p", "rb"))

{'domains__art_ASC': -3435.0034007452377,
 'domains__art__age': -0.12202041213932821,
 'domains__art__behaviors__chill': 1.25326112196447,
 'domains__art__behaviors__collect': 0.08391240023302936,
 'domains__art__behaviors__combat': 0.09345005181309078,
 'domains__art__behaviors__compete': -0.36292305997656016,
 'domains__art__behaviors__complete': -0.6616076839606051,
 'domains__art__behaviors__cooperate': 0.5230593414469284,
 'domains__art__behaviors__destroy': 0.8180822082374998,
 'domains__art__behaviors__discover': 2.3653557053896526,
 'domains__art__behaviors__empower': 0.0720227512567071,
 'domains__art__behaviors__experiment': 0.25791101786547777,
 'domains__art__behaviors__express': -0.5922546955523607,
 'domains__art__behaviors__immerse': 0.4746910023211322,
 'domains__art__behaviors__plan': 0.5847625288707279,
 'domains__art__behaviors__react': -0.6752669019769335,
 'domains__art__behaviors__solve': 0.9199550013674176,
 'domains__art__genre__arcade': 1.6105005350509989,
 'do

# Postprocess & save data

In [28]:
GENRE_NA_COLS = [
    'genre__building_na',
    'genre__driving_na',
    'genre__education_na',
    'genre__idle_na',
    'genre__party_na',
    'genre__sports_na',
    'genre__story_na',
    'genre__tabletop_na'
]

In [29]:
mechanic_coefs = pickle.load(open("mechanic_coefs_2.p", "rb"))
entity_coefs = pickle.load(open("entity_coefs_2.p", "rb"))
role_coefs = pickle.load(open("role_coefs_2.p", "rb"))
enviro_era_coefs = pickle.load(open("enviro_era_coefs_2.p", "rb"))
focus_domain_coefs = pickle.load(open("focus_domain_coefs_2.p", "rb"))

In [30]:
all_coefs = {**mechanic_coefs, **entity_coefs, **role_coefs, **enviro_era_coefs, **focus_domain_coefs}

In [31]:
{x: y for x, y in all_coefs.items() if 'gender' in x}

{}

In [32]:
# replace _ASC with __base
all_coefs = {(x[:-4] + '__base' if x.endswith('_ASC') else x): y for x, y in all_coefs.items()}

In [33]:
coefs_df = pd.DataFrame(all_coefs, index=['value']).T.reset_index()
coefs_df = coefs_df.rename(columns={'index': 'combined_col'})
coefs_df

,combined_col,value
0,core__4x__base,-0.288217
1,core__4x__age,-0.269079
2,core__4x__behaviors__chill,-0.721520
3,core__4x__behaviors__collect,-0.445766
4,core__4x__behaviors__combat,0.591848
...,...,...
9736,focus__weapons__topics__horror,0.911709
9737,focus__weapons__topics__mystery,0.296231
9738,focus__weapons__topics__realistic,-0.598322
9739,focus__weapons__topics__romance,-1.859022


In [34]:
coefs_df['feature'] = coefs_df.combined_col.apply(lambda x: '__'.join(x.split('__')[:2]) if '___' not in x else ('__'.join(x.split('__')[:2])) + '_')
coefs_df['user_col'] = coefs_df.combined_col.apply(lambda x: '__'.join(x.split('__')[2:]) if '___' not in x else ('__'.join(x.split('__')[2:]))[1:])
coefs_df

,combined_col,value,feature,user_col
0,core__4x__base,-0.288217,core__4x,base
1,core__4x__age,-0.269079,core__4x,age
2,core__4x__behaviors__chill,-0.721520,core__4x,behaviors__chill
3,core__4x__behaviors__collect,-0.445766,core__4x,behaviors__collect
4,core__4x__behaviors__combat,0.591848,core__4x,behaviors__combat
...,...,...,...,...
9736,focus__weapons__topics__horror,0.911709,focus__weapons,topics__horror
9737,focus__weapons__topics__mystery,0.296231,focus__weapons,topics__mystery
9738,focus__weapons__topics__realistic,-0.598322,focus__weapons,topics__realistic
9739,focus__weapons__topics__romance,-1.859022,focus__weapons,topics__romance


In [35]:
coefs_df.user_col.unique()

array(['base', 'age', 'behaviors__chill', 'behaviors__collect',
       'behaviors__combat', 'behaviors__compete', 'behaviors__complete',
       'behaviors__cooperate', 'behaviors__destroy',
       'behaviors__discover', 'behaviors__empower',
       'behaviors__experiment', 'behaviors__express',
       'behaviors__immerse', 'behaviors__plan', 'behaviors__react',
       'behaviors__solve', 'genre__arcade', 'genre__building',
       'genre__building_na', 'genre__casino', 'genre__combat',
       'genre__driving', 'genre__driving_na', 'genre__education',
       'genre__education_na', 'genre__idle', 'genre__idle_na',
       'genre__match', 'genre__party', 'genre__party_na', 'genre__puzzle',
       'genre__rpg', 'genre__simulation', 'genre__sports',
       'genre__sports_na', 'genre__story', 'genre__story_na',
       'genre__strategy', 'genre__tabletop', 'genre__tabletop_na',
       'topics__action', 'topics__adventure', 'topics__comedy',
       'topics__drama', 'topics__fantasy', 'topics__ho

In [36]:
coefs_p_df = coefs_df.pivot(
    index="feature", columns="user_col", values="value"
)

coefs_p_df.index.name = None
coefs_p_df.columns.name = None

coefs_p_df = coefs_p_df[['base', 'age'] + BEHAVIOUR_COLS + GENRE_COLS + GENRE_NA_COLS + TOPIC_COLS] #gender_female
coefs_p_df

,base,age,behaviors__chill,behaviors__collect,behaviors__combat,behaviors__compete,behaviors__complete,behaviors__cooperate,behaviors__destroy,behaviors__discover,...,topics__action,topics__adventure,topics__comedy,topics__drama,topics__fantasy,topics__horror,topics__mystery,topics__realistic,topics__romance,topics__sci_fi
core__4x,-0.288217,-0.269079,-0.721520,-0.445766,0.591848,-0.273289,-0.045995,-0.087741,0.056898,0.317867,...,0.150128,0.133865,-1.158049,0.074926,0.821655,0.088780,0.276168,0.238102,1.050658,0.866008
core__agriculture,-2.095988,3.149500,4.223846,-2.130348,0.783652,-0.829428,6.280429,-3.217267,-0.366344,-1.695354,...,0.519221,-4.713135,-1.636614,-3.874839,-3.050578,3.238996,2.405990,-1.588836,5.940052,2.355202
core__artillery,0.221650,2.887373,1.055477,-2.779875,-0.982867,0.113594,-2.203189,-0.130615,0.599708,1.345202,...,2.522744,-0.506594,-0.247778,-1.433856,3.924468,-0.402212,-2.738111,1.748354,2.228789,0.243054
core__asymetric,0.077696,-1.169540,0.064858,0.683366,2.218087,1.994487,-1.111897,-0.855222,1.077095,-1.093263,...,-1.965813,-1.915937,-1.082739,1.394624,-0.274317,0.822699,2.357365,2.005242,-0.012946,1.058445
core__auto_chess,-0.346815,0.567837,0.053205,-0.104938,-0.045398,-0.246239,-0.779069,0.047883,0.438725,-1.213328,...,0.565337,-0.101891,-0.784482,0.267178,0.526424,0.027829,-0.083057,1.073358,0.693864,0.375522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
roles__rebel,0.751686,0.655068,0.184663,0.388596,0.351097,0.395381,0.229533,0.632679,0.518670,1.676821,...,-0.108339,0.380862,0.107868,-0.267716,0.193756,0.896328,0.462004,0.208374,0.489269,0.707283
roles__solver,0.964600,0.152337,0.557487,-0.272942,0.456544,-0.111690,-1.631243,0.885063,0.456824,1.582681,...,-0.942705,0.028290,-0.328575,-0.853055,0.579779,-0.121151,1.504823,0.578005,0.138080,1.540291
roles__team,0.083764,1.284258,0.855827,1.302641,-0.454301,0.618478,-0.389286,1.107102,0.398236,0.174405,...,0.472611,0.880932,-0.011424,0.392401,0.850078,0.750251,-0.137671,0.413312,-0.937953,0.661336
roles__villain,-1.135686,0.228337,0.659040,-0.136965,0.639805,0.467726,-1.251640,0.705736,0.448906,0.870515,...,-0.050766,0.743828,-0.389327,0.475194,0.991958,1.296251,-0.264078,0.341225,0.356898,1.390100


In [37]:
coefs_p_df.reset_index()

,index,base,age,behaviors__chill,behaviors__collect,behaviors__combat,behaviors__compete,behaviors__complete,behaviors__cooperate,behaviors__destroy,...,topics__action,topics__adventure,topics__comedy,topics__drama,topics__fantasy,topics__horror,topics__mystery,topics__realistic,topics__romance,topics__sci_fi
0,core__4x,-0.288217,-0.269079,-0.721520,-0.445766,0.591848,-0.273289,-0.045995,-0.087741,0.056898,...,0.150128,0.133865,-1.158049,0.074926,0.821655,0.088780,0.276168,0.238102,1.050658,0.866008
1,core__agriculture,-2.095988,3.149500,4.223846,-2.130348,0.783652,-0.829428,6.280429,-3.217267,-0.366344,...,0.519221,-4.713135,-1.636614,-3.874839,-3.050578,3.238996,2.405990,-1.588836,5.940052,2.355202
2,core__artillery,0.221650,2.887373,1.055477,-2.779875,-0.982867,0.113594,-2.203189,-0.130615,0.599708,...,2.522744,-0.506594,-0.247778,-1.433856,3.924468,-0.402212,-2.738111,1.748354,2.228789,0.243054
3,core__asymetric,0.077696,-1.169540,0.064858,0.683366,2.218087,1.994487,-1.111897,-0.855222,1.077095,...,-1.965813,-1.915937,-1.082739,1.394624,-0.274317,0.822699,2.357365,2.005242,-0.012946,1.058445
4,core__auto_chess,-0.346815,0.567837,0.053205,-0.104938,-0.045398,-0.246239,-0.779069,0.047883,0.438725,...,0.565337,-0.101891,-0.784482,0.267178,0.526424,0.027829,-0.083057,1.073358,0.693864,0.375522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186,roles__rebel,0.751686,0.655068,0.184663,0.388596,0.351097,0.395381,0.229533,0.632679,0.518670,...,-0.108339,0.380862,0.107868,-0.267716,0.193756,0.896328,0.462004,0.208374,0.489269,0.707283
187,roles__solver,0.964600,0.152337,0.557487,-0.272942,0.456544,-0.111690,-1.631243,0.885063,0.456824,...,-0.942705,0.028290,-0.328575,-0.853055,0.579779,-0.121151,1.504823,0.578005,0.138080,1.540291
188,roles__team,0.083764,1.284258,0.855827,1.302641,-0.454301,0.618478,-0.389286,1.107102,0.398236,...,0.472611,0.880932,-0.011424,0.392401,0.850078,0.750251,-0.137671,0.413312,-0.937953,0.661336
189,roles__villain,-1.135686,0.228337,0.659040,-0.136965,0.639805,0.467726,-1.251640,0.705736,0.448906,...,-0.050766,0.743828,-0.389327,0.475194,0.991958,1.296251,-0.264078,0.341225,0.356898,1.390100


### Check coefs for all users

In [38]:
coef_cols = []

for col in coefs_p_df.index:
    coef_col = pd.Series([coefs_p_df.loc[col, 'base']]*len(user_df), name=col)
    
    for user_col in coefs_p_df.columns[1:]:
        user_col_min = user_df[user_col].min()
        user_col_max = user_df[user_col].max()
        
        # min-max scaling -> 0-1
        coef_col += ((user_df[user_col] - user_col_min) / (user_col_max - user_col_min)) * coefs_p_df.loc[col, user_col]
    
    coef_cols.append(coef_col)

In [39]:
test_df = pd.concat(coef_cols, axis=1)
test_df['survey_instance'] = user_df['survey_instance']

In [40]:
test_df['gender_female'] = user_df['gender_female']

In [41]:
gender_df = test_df.groupby('gender_female').mean().T
gender_df['category'] = gender_df.index.to_series().apply(lambda x: x.split('__')[0])
category_df = gender_df.groupby('category').mean()
category_gender_boost_dict = (category_df[0] - category_df[1]).to_dict()

In [42]:
category_gender_boost_dict

{'core': 0.4592130914385406,
 'domains': 1.05912611533794,
 'entities': 1.3250699504754553,
 'environment': 0.7538794755434801,
 'eras': 1.0339127471304197,
 'focus': 1.132667886078633,
 'roles': 1.1079398004491734}

In [43]:
coefs_p_df['gender_female'] = gender_df.category.apply(lambda x: category_gender_boost_dict[x])

In [44]:
coefs_p_df

,base,age,behaviors__chill,behaviors__collect,behaviors__combat,behaviors__compete,behaviors__complete,behaviors__cooperate,behaviors__destroy,behaviors__discover,...,topics__adventure,topics__comedy,topics__drama,topics__fantasy,topics__horror,topics__mystery,topics__realistic,topics__romance,topics__sci_fi,gender_female
core__4x,-0.288217,-0.269079,-0.721520,-0.445766,0.591848,-0.273289,-0.045995,-0.087741,0.056898,0.317867,...,0.133865,-1.158049,0.074926,0.821655,0.088780,0.276168,0.238102,1.050658,0.866008,0.459213
core__agriculture,-2.095988,3.149500,4.223846,-2.130348,0.783652,-0.829428,6.280429,-3.217267,-0.366344,-1.695354,...,-4.713135,-1.636614,-3.874839,-3.050578,3.238996,2.405990,-1.588836,5.940052,2.355202,0.459213
core__artillery,0.221650,2.887373,1.055477,-2.779875,-0.982867,0.113594,-2.203189,-0.130615,0.599708,1.345202,...,-0.506594,-0.247778,-1.433856,3.924468,-0.402212,-2.738111,1.748354,2.228789,0.243054,0.459213
core__asymetric,0.077696,-1.169540,0.064858,0.683366,2.218087,1.994487,-1.111897,-0.855222,1.077095,-1.093263,...,-1.915937,-1.082739,1.394624,-0.274317,0.822699,2.357365,2.005242,-0.012946,1.058445,0.459213
core__auto_chess,-0.346815,0.567837,0.053205,-0.104938,-0.045398,-0.246239,-0.779069,0.047883,0.438725,-1.213328,...,-0.101891,-0.784482,0.267178,0.526424,0.027829,-0.083057,1.073358,0.693864,0.375522,0.459213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
roles__rebel,0.751686,0.655068,0.184663,0.388596,0.351097,0.395381,0.229533,0.632679,0.518670,1.676821,...,0.380862,0.107868,-0.267716,0.193756,0.896328,0.462004,0.208374,0.489269,0.707283,1.107940
roles__solver,0.964600,0.152337,0.557487,-0.272942,0.456544,-0.111690,-1.631243,0.885063,0.456824,1.582681,...,0.028290,-0.328575,-0.853055,0.579779,-0.121151,1.504823,0.578005,0.138080,1.540291,1.107940
roles__team,0.083764,1.284258,0.855827,1.302641,-0.454301,0.618478,-0.389286,1.107102,0.398236,0.174405,...,0.880932,-0.011424,0.392401,0.850078,0.750251,-0.137671,0.413312,-0.937953,0.661336,1.107940
roles__villain,-1.135686,0.228337,0.659040,-0.136965,0.639805,0.467726,-1.251640,0.705736,0.448906,0.870515,...,0.743828,-0.389327,0.475194,0.991958,1.296251,-0.264078,0.341225,0.356898,1.390100,1.107940


In [46]:
test_df

,core__4x,core__agriculture,core__artillery,core__asymetric,core__auto_chess,core__automation,core__backtracking,core__ball,core__baseball,core__basketball,...,roles__explorer,roles__follower,roles__mystic,roles__rebel,roles__solver,roles__team,roles__villain,roles__warrior,survey_instance,gender_female
0,2.754849,1.041384,1.068867,0.946033,1.147250,0.141344,2.237014,1.824652,0.589785,-2.872531,...,1.933209,1.602612,2.145300,1.671991,1.220235,1.121867,1.606129,1.855041,33cbe449-af6a-484b-a521-6a4d08f73454,1
1,2.518849,6.859815,1.514870,4.370928,3.450785,-0.408321,4.274114,2.172749,3.731515,8.392953,...,3.924299,2.051001,3.591273,2.847747,2.371497,2.031164,2.554284,2.151207,5db16995-594b-4b4e-9151-e502b85d6ff3,0
2,-0.550507,-10.337764,6.668011,-5.697838,0.873933,3.758818,-1.163995,1.418757,4.971516,-1.615916,...,-1.257699,0.432299,0.643819,-1.969505,-0.380333,0.648706,0.637378,0.786648,9722c487-9a13-4a9d-8599-577b69805034,1
3,3.316473,17.242549,3.187699,3.523599,2.622949,3.321279,3.233665,3.085486,-5.487018,11.536411,...,3.222076,2.880099,4.002295,3.790279,3.523843,2.752532,2.958466,4.576350,7094ec2f-88d4-4642-90f6-f12b41eca4f1,0
4,2.680931,2.961865,5.807817,3.795726,1.878644,-1.659182,3.523439,3.714703,3.400149,3.641580,...,3.884730,2.199074,2.534741,2.178480,1.961275,2.752727,2.156843,3.298265,eb6a4d45-c680-4eaf-bef7-aebe5edb7ce0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1965,4.178016,5.803082,1.591647,1.916382,3.616533,4.585763,2.480446,3.108865,-0.935953,-3.349519,...,2.996723,2.147984,3.180166,2.668171,2.455445,2.268175,2.543349,2.300299,35fc0bbf-cef7-4ad9-b0c7-8bfb50043cc3,0
1966,1.439444,1.221211,-6.904302,0.046238,-0.216698,0.367566,2.663422,0.420181,0.656237,-0.663239,...,-0.305527,-1.340316,-1.606243,-0.021790,0.546096,0.020058,0.422941,-0.349778,b11b6d9c-f739-478a-ab26-d26d11825827,0
1967,3.113234,3.424641,-3.125021,-0.246251,0.938250,1.390719,1.381462,2.269673,-2.976818,1.837184,...,2.626441,2.044420,1.601704,2.251623,3.599893,1.862688,2.282003,3.300912,e9c3e858-826f-479e-a85d-6ea3b0e4e43a,0
1968,1.377966,-3.902136,2.224523,-5.089354,1.928476,3.213606,2.594824,1.194597,-8.501987,2.497875,...,1.479377,0.860811,1.901919,2.053584,0.296282,2.680320,1.678849,3.574843,b6e5ec83-0b04-4f2d-8d54-1bc0d6785adc,0


In [56]:
coef_cols = []

for col in coefs_p_df.index:
    coef_col = pd.Series([coefs_p_df.loc[col, 'base']]*len(user_df), name=col)
    
    for user_col in coefs_p_df.columns[1:]:
        user_col_min = user_df[user_col].min()
        user_col_max = user_df[user_col].max()
        
        # min-max scaling -> 0-1
        coef_col += ((user_df[user_col] - user_col_min) / (user_col_max - user_col_min)) * coefs_p_df.loc[col, user_col]
    
    coef_cols.append(coef_col)

In [57]:
test_df = pd.concat(coef_cols, axis=1)
test_df['survey_instance'] = user_df['survey_instance']

In [50]:
test_df['gender_female'] = user_df['gender_female']

In [55]:
with pd.option_context('display.max_columns', 500):
    display(test_df.groupby('gender_female').mean())

,core__4x,core__agriculture,core__artillery,core__asymetric,core__auto_chess,core__automation,core__backtracking,core__ball,core__baseball,core__basketball,core__betrayal,core__bingo,core__blackjack,core__blast,core__board,core__brain,core__branched_story,core__brawl,core__bubble,core__bullet_hell,core__cards,core__caring,core__ccg,core__chess,core__coloring,core__combos,core__connect,core__construct,core__control_time,core__cover_fire,core__crafting,core__crush_bricks,core__dare,core__dating,core__deathmatch,core__design,core__dialogue_tree,core__dice,core__differences,core__domino,core__dress_up,core__escape_room,core__esports,core__fighter,core__fishing,core__flying,core__football,core__full_motion_video,core__golf,core__grow_wealth,core__guess,core__hand_to_hand,core__hidden_object,core__hockey,core__hunting,core__jigsaw,core__karaoke,core__klotski,core__lane,core__legion,core__life,core__linear_story,core__loftto,core__luck_battle,core__mahjong,core__manage_inventory,core__management,core__martial,core__match_tile,core__maze,core__melee,core__merge,core__minesweeper,core__mini_games,core__mmo,core__moba,core__moral_choices,core__move_blocks,core__nonogram,core__okey,core__open_world,core__parkour,core__physics,core__platformer,core__poker,core__push_coin,core__racing,core__random_loot,core__restore,core__rhythm,core__road_driving,core__roguelike,core__royale,core__runner,core__sailing,core__sandbox,core__shark,core__skating,core__skill_tree,core__slicing,core__slots,core__sniper,core__soccer,core__solitaire,core__sort water,core__stealth,core__sudoku,core__summon,core__survival,core__swapping,core__switch,core__tactical,core__taming,core__telekinesis,core__tennis,core__time_attack,core__time_loop,core__tower_defense,core__trading,core__trivia,core__upgrading,core__vehicular,core__word,domains__art,domains__business,domains__crime,domains__family,domains__fashion,domains__food,domains__health,domains__luxury,domains__military,domains__mythology,domains__nature,domains__politics,domains__religion,domains__rights,domains__sports,entities__adult,entities__alien,entities__animal,entities__celestial,entities__elder,entities__human,entities__kid,entities__microbe,entities__monster,entities__plant,entities__robot,entities__super_,entities__zombie,environment__alternality,environment__aquatic,environment__rural,environment__space,environment__suburban,environment__urban,environment__virtuality,environment__wilderness,eras__ancient,eras__industrial,eras__medieval,eras__modern,eras__near_future,eras__prehistoric,eras__renaissance,eras__retro,eras__timeless,focus__actors,focus__bases,focus__cities,focus__events,focus__interiors,focus__levels,focus__objects,focus__self,focus__solutions,focus__structures,focus__vehicles,focus__weapons,roles__authority,roles__carer,roles__creator,roles__explorer,roles__follower,roles__mystic,roles__rebel,roles__solver,roles__team,roles__villain,roles__warrior
gender_female,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,1.913623,2.288986,2.039414,0.901697,1.562333,1.288123,1.828587,1.714475,-2.335257,1.534117,-1.517954,1.827774,2.035344,2.082371,1.868619,2.076658,3.846240,1.870364,2.164456,1.554646,0.916450,1.677375,2.315219,1.821980,0.255419,1.497009,2.122965,1.474276,1.667517,1.468424,1.691395,1.603547,2.078026,0.007089,2.209160,1.893465,2.605522,3.379457,1.820804,1.795178,0.316120,2.179821,1.837648,1.465432,1.226890,2.180447,2.697344,6.343316,0.930127,2.635924,4.399781,1.836525,1.810475,1.613560,1.298913,1.676347,3.629652,1.921278,1.818499,1.739633,2.025198,2.989233,1.559443,3.465340,4.403665,1.499283,1.984165,0.322203,2.030280,1.880257,1.116858,2.316742,1.175651,2.925398,1.338443,1.323269,15.234529,1.855331,1.764527,1.301345,2.166382,1.779508,1.836115,1.733423,2.096029,2.042813,-9.240824,2.033297,3.633605,1.582601,3.851346,0.435413,1.363227,1.787261,-2.5808

In [58]:
# df_guid = '057c206c-42b5-4f6d-915d-4f772f13e761'
# df_guid = 'be8b281a-5fa9-4308-8b23-0f1ff92f1070'
df_guid = '0ce01ac6-d0b0-4b79-b89c-a68f4c7a783b'

In [64]:
write_df_to_db(conn, test_df, df_guid)

In [65]:
conn.close()

In [72]:
test_df.mean().to_dict()

{'core__4x': 1.7024892484165792,
 'core__agriculture': 2.5886451886238535,
 'core__artillery': 1.8015088940091166,
 'core__asymetric': 0.6030392729400657,
 'core__auto_chess': 1.543783499788681,
 'core__automation': 1.5088269251738249,
 'core__backtracking': 1.4606976421753353,
 'core__ball': 1.5433171668091503,
 'core__baseball': -10.558695286787964,
 'core__basketball': 0.8492632946642641,
 'core__betrayal': -3.2205453719609975,
 'core__bingo': 2.094342594593271,
 'core__blackjack': 1.9992802769711997,
 'core__blast': 1.9063204632896393,
 'core__board': -1.9265158861488378,
 'core__brain': 2.2940678343952174,
 'core__branched_story': 0.9492715399813214,
 'core__brawl': 1.4896816496463638,
 'core__bubble': 2.206965953559486,
 'core__bullet_hell': 1.2738227995311642,
 'core__cards': 1.5011162122865584,
 'core__caring': 1.805482057149928,
 'core__ccg': 3.0438508251479925,
 'core__chess': 3.890170351039853,
 'core__coloring': -0.40890082057816735,
 'core__combos': 2.089763097713083,
 'co

In [73]:
test_df.std().to_dict()

{'core__4x': 1.6023195395664127,
 'core__agriculture': 4.038787846063806,
 'core__artillery': 5.156294753562237,
 'core__asymetric': 3.4036516192031456,
 'core__auto_chess': 1.5615137341315917,
 'core__automation': 4.145596343959279,
 'core__backtracking': 1.8883715533110081,
 'core__ball': 1.656460314441771,
 'core__baseball': 16.050280969275118,
 'core__basketball': 13.305495388072883,
 'core__betrayal': 21.68240958444509,
 'core__bingo': 0.8722477114527342,
 'core__blackjack': 1.2478444370280077,
 'core__blast': 1.3040943602362864,
 'core__board': 18.892831974338986,
 'core__brain': 1.0692101280519934,
 'core__branched_story': 5.7182815741425035,
 'core__brawl': 2.7308743239487856,
 'core__bubble': 0.7444383787271265,
 'core__bullet_hell': 3.3824291862166085,
 'core__cards': 11.171155003648808,
 'core__caring': 3.0398483119434156,
 'core__ccg': 7.594419659105374,
 'core__chess': 7.606160804189169,
 'core__coloring': 16.374273888076576,
 'core__combos': 4.824936916449704,
 'core__con

### Save data

In [63]:
credentials = {
  "user": "external.data.user",
  "password": "wIfQxcjgZ5!gA*FWgi1z$Kj9",
  "host": "35.208.148.42",
  "port": 3306
}

conn: MySQLConnection = mysql.connector.connect(**credentials)

In [61]:
def write_df_to_db(conn: MySQLConnection, df: pd.DataFrame, guid: str):
    buf = io.BytesIO()
    df.to_feather(buf)
    buf.seek(0)
    b = buf.read()

    cur = conn.cursor()
    cur.execute("""
    DELETE FROM external_data.dms WHERE guid = %s
    """, [guid])
    cur.execute("""
    INSERT INTO external_data.dms (guid, data) VALUES (%s, %s)
    """, [guid, b])
    cur.close()
    conn.commit()

In [13]:
df_guid = '7de9c732-cb01-421c-b2ac-c77a88d9f965'

In [102]:
write_df_to_db(conn, coefs_p_df.reset_index(), df_guid)

In [103]:
conn.close()